## Caching — why, and the strategies

A database query takes a **millisecond at best, hundreds under load**; an in-memory cache lookup takes **microseconds** — three or four orders of magnitude faster. For any read pattern where the same answer is computed over and over, **putting a cache in front of the database is the single highest-leverage performance change** on AWS. The shape: the app checks the cache first — **hit** returns immediately; **miss** falls through to the DB, writes the result into the cache, then returns — so subsequent reads within the **TTL** get the fast path. AWS offers **ElastiCache** (managed Redis/Valkey/Memcached — general-purpose, in front of anything) and **DAX** (a write-through cache purpose-built for DynamoDB *only*, drop-in for the client).

The patterns worth knowing: **Lazy loading (cache-aside)** — the app reads cache, on miss reads DB and writes back; the cache holds only what's read and a cache outage just slows things (falls through to DB), but the **first read is always a miss** and data can go **stale**. **Write-through** — every write goes to **both** cache and DB, so reads never miss written data and the cache stays consistent, at the cost of a **double-write** and caching data that may never be read. **TTL** — every entry carries an expiry, the simple bound on staleness, and pairs with lazy loading to cap how long stale data lives.

Most designs **combine lazy loading + TTL** (cache-aside with an expiry); add **write-through** only where read-after-write on cached data matters.